# Complete ML Pipeline: Causal Inference with LLM Embeddings

## What This Notebook Does

This notebook demonstrates how to use **Large Language Model (LLM) text embeddings** as proxies for unobserved confounders in causal inference. We generate realistic freelancer data using **Google's Gemini API**, extract text embeddings, and apply multiple causal estimation methods.

### Key Concepts for Beginners

1. **Causal Inference**: We want to estimate the TRUE effect of a program on earnings, not just a correlation
2. **Confounding**: High-ability workers both earn more AND are more likely to join the program, creating bias
3. **Text as Proxy**: Profile text reveals "ability" - we can use embeddings to control for this latent confounder
4. **Double Machine Learning (DML)**: A modern method that uses ML to control for high-dimensional confounders

### Pipeline Overview

```
Step 1: Generate Data (Gemini API)  →  Realistic freelancer profiles with known causal effect
Step 2: Extract Embeddings          →  Convert text to 384-dimensional vectors
Step 3: Dimensionality Reduction    →  PCA to compress embeddings
Step 4: Naive Estimation            →  Baseline (biased) estimate
Step 5: DML Estimation              →  Bias-corrected estimate using embeddings
Step 6: Bayesian Estimation         →  Full uncertainty quantification
Step 7: Compare Results             →  Validate against ground truth ($5.00)
```

### Ground Truth

**TRUE CAUSAL EFFECT = $5.00/hour**

We validate all methods against this known ground truth.

---

## SECTION 1: Configuration & Setup

### Prerequisites

Before running this notebook, you need:

1. **Gemini API Key** (FREE):
   - Visit: https://aistudio.google.com/app/apikey
   - Create a new API key
   - Create a `.env` file in the project root with: `GEMINI_API_KEY=your_key_here`

2. **Python packages** (install if needed):
   ```bash
   pip install pandas numpy scipy scikit-learn sentence-transformers
   pip install bambi pymc arviz matplotlib seaborn
   pip install python-dotenv requests tqdm
   ```

### Configuration Parameters

Adjust these parameters based on your needs:

In [ ]:
# =============================================================================
# CONFIGURATION - Adjust these parameters as needed
# =============================================================================

# --- Data Generation Settings ---
N_SAMPLES = 500              # Number of freelancers to generate
                              # Start small (100-500) for testing, use 5000 for full analysis
                              # Each sample requires one Gemini API call (~2 seconds)

RANDOM_SEED = 42              # For reproducibility - same seed = same results

# --- Gemini API Settings ---
USE_GEMINI_API = True         # Set to True to use Gemini API for realistic profiles
                              # Set to False to use fast template-based fallback (for testing)

API_BATCH_SIZE = 50           # Pause every N requests to avoid rate limiting
API_RETRY_ATTEMPTS = 3        # Number of retries on API failure

# --- Embedding Settings ---
EMBEDDING_MODEL = 'all-MiniLM-L6-v2'  # Pre-trained sentence transformer model
                                       # Options: 'all-MiniLM-L6-v2' (384d, fast)
                                       #          'all-mpnet-base-v2' (768d, better quality)

# --- PCA Settings ---
PCA_VARIANCE_THRESHOLD = 0.90  # Keep components explaining this much variance
MAX_PCA_COMPONENTS = 50        # Maximum PCA components to keep

# --- Causal Estimation Settings ---
DML_CV_FOLDS = 5               # Cross-validation folds for DML
DML_N_PCA_COMPONENTS = 20      # Number of PCA components for DML (Random Forest variant)

# --- Bayesian Settings ---
BAYES_N_PCA_COMPONENTS = 10    # Number of PCA components for Bayesian model (keep small)
MCMC_DRAWS = 2000              # Number of MCMC posterior draws
MCMC_TUNE = 1000               # Number of tuning steps (burn-in)
HDI_PROB = 0.94                # Highest Density Interval probability

# --- Ground Truth (DO NOT CHANGE) ---
TRUE_CAUSAL_EFFECT = 5.0       # The true treatment effect we're trying to recover

# --- File Paths ---
PROJECT_DIR = '/home/user/Causal---Embeddings-'
DATA_OUTPUT_PATH = f'{PROJECT_DIR}/synthetic_upwork_data.parquet'
EMBEDDINGS_OUTPUT_PATH = f'{PROJECT_DIR}/data_with_embeddings.parquet'
RESULTS_OUTPUT_PATH = f'{PROJECT_DIR}/causal_estimates.csv'

print("Configuration loaded successfully!")
print(f"  - Sample size: {N_SAMPLES}")
print(f"  - Use Gemini API: {USE_GEMINI_API}")
print(f"  - Embedding model: {EMBEDDING_MODEL}")
print(f"  - True causal effect: ${TRUE_CAUSAL_EFFECT:.2f}")

In [ ]:
# =============================================================================
# IMPORTS - Load all required libraries
# =============================================================================

# Standard data science libraries
import pandas as pd              # DataFrames for tabular data
import numpy as np               # Numerical computing
import matplotlib.pyplot as plt  # Plotting
import seaborn as sns            # Statistical visualizations
from pathlib import Path         # File path handling
import warnings
warnings.filterwarnings('ignore')  # Suppress non-critical warnings

# Statistical functions
from scipy.special import expit  # Sigmoid function for propensity scores
from scipy import stats          # Statistical tests

# Text embeddings
from sentence_transformers import SentenceTransformer  # Pre-trained language model

# Machine learning
from sklearn.decomposition import PCA           # Dimensionality reduction
from sklearn.preprocessing import StandardScaler  # Feature standardization
from sklearn.linear_model import LassoCV, LinearRegression  # Regression models
from sklearn.ensemble import RandomForestRegressor  # Non-linear model
from sklearn.model_selection import cross_val_predict  # Cross-validation

# Bayesian modeling
import bambi as bmb              # Bayesian model building interface
import arviz as az               # Bayesian analysis and visualization

# Utilities
import joblib                    # Save/load models
from datetime import datetime    # Timestamps
import sys
import os
import time
import requests                  # HTTP requests for API calls
import json                      # JSON parsing
from tqdm.notebook import tqdm   # Progress bars
from dotenv import load_dotenv   # Load environment variables

# Set random seed for reproducibility
np.random.seed(RANDOM_SEED)

# Plot styling
plt.style.use('seaborn-v0_8-darkgrid')
sns.set_palette("husl")

# Add project directory to path (to import local modules)
sys.path.insert(0, PROJECT_DIR)

print("✓ All libraries imported successfully!")

---

## SECTION 2: Data Generation with Gemini API

### How It Works

1. **Structured Data Generation**: Create demographics, platform metrics, treatment assignment, and outcomes
2. **Text Profile Generation**: Use Gemini API to generate realistic profile text based on ability level
3. **Causal Structure**: The data has a known causal effect ($5.00) with strong confounding

### The Confounding Problem

```
                    Ability (U)
                   /           \
                  v             v
        Program Participation  →  Hourly Earnings
              (Treatment)           (Outcome)
```

High-ability freelancers:
- Are MORE likely to join the program (selection bias)
- Earn MORE regardless of the program (confounding)

This creates severe upward bias in naive estimates!

In [ ]:
# =============================================================================
# LOAD GEMINI API KEY
# =============================================================================

# Load environment variables from .env file
load_dotenv()

# Get the API key
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')

# Check if API key is configured
if USE_GEMINI_API:
    if not GEMINI_API_KEY or GEMINI_API_KEY == 'your_api_key_here':
        print("⚠️  WARNING: Gemini API key not configured!")
        print("")
        print("To use Gemini API for realistic profile generation:")
        print("1. Visit: https://aistudio.google.com/app/apikey")
        print("2. Create a FREE API key")
        print("3. Create a file named '.env' in the project root")
        print("4. Add this line: GEMINI_API_KEY=your_actual_key_here")
        print("")
        print("For now, falling back to template-based generation...")
        USE_GEMINI_API = False
    else:
        print(f"✓ Gemini API key loaded: {GEMINI_API_KEY[:8]}...{GEMINI_API_KEY[-4:]}")
        # Build the API URL
        GEMINI_API_URL = f"https://generativelanguage.googleapis.com/v1beta/models/gemini-1.5-flash:generateContent?key={GEMINI_API_KEY}"
else:
    print("ℹ️  Using template-based generation (Gemini API disabled)")

In [ ]:
# =============================================================================
# DATA GENERATION CONSTANTS
# =============================================================================

# Egyptian cities (our study population)
EGYPTIAN_CITIES = [
    'Cairo', 'Alexandria', 'Giza', 'Shubra El-Kheima', 'Port Said',
    'Suez', 'Luxor', 'Mansoura', 'El-Mahalla El-Kubra', 'Tanta',
    'Asyut', 'Ismailia', 'Fayyum', 'Zagazig', 'Aswan', 'Damietta'
]

# Freelancer categories
CATEGORIES = [
    'Web Development', 'Graphic Design', 'Translation',
    'Administrative Support', 'Mobile App Development', 'Content Writing',
    'Digital Marketing', 'Video Editing', 'Data Entry', 'SEO Specialist',
    'UI/UX Design', 'Social Media Management', 'Accounting', 'Customer Support'
]

# Education levels
EDUCATION_LEVELS = [
    'High School', 'Some College', "Bachelor's Degree",
    "Master's Degree", 'Professional Certificate', 'Self-Taught'
]

# Category demand scores (affects earnings)
CATEGORY_DEMAND = {
    'Web Development': 0.9, 'Mobile App Development': 0.85, 'UI/UX Design': 0.8,
    'Digital Marketing': 0.75, 'SEO Specialist': 0.7, 'Graphic Design': 0.65,
    'Content Writing': 0.6, 'Translation': 0.55, 'Video Editing': 0.6,
    'Data Entry': 0.4, 'Administrative Support': 0.45, 'Customer Support': 0.5,
    'Social Media Management': 0.65, 'Accounting': 0.55
}

# Category effects on earnings
CATEGORY_EFFECTS = {
    'Web Development': 8, 'Mobile App Development': 10, 'UI/UX Design': 7,
    'Digital Marketing': 5, 'SEO Specialist': 4, 'Graphic Design': 4,
    'Content Writing': 3, 'Translation': 3, 'Video Editing': 5,
    'Data Entry': -2, 'Administrative Support': -1, 'Customer Support': 0,
    'Social Media Management': 4, 'Accounting': 6
}

print(f"✓ Data generation constants loaded")
print(f"  - {len(EGYPTIAN_CITIES)} cities")
print(f"  - {len(CATEGORIES)} categories")
print(f"  - {len(EDUCATION_LEVELS)} education levels")

In [ ]:
# =============================================================================
# FUNCTION: Generate Structured Data
# =============================================================================

def generate_structured_data(n: int, seed: int = RANDOM_SEED) -> pd.DataFrame:
    """
    Generate the structured variables for the synthetic dataset.
    
    This function creates a dataset with known causal structure:
    - Latent ability (U) affects both treatment and outcome
    - Treatment effect is exactly $5.00 (our ground truth)
    - Strong selection bias: high-ability → more likely to participate
    
    Parameters:
    -----------
    n : int
        Number of observations to generate
    seed : int
        Random seed for reproducibility
        
    Returns:
    --------
    pd.DataFrame
        Dataset with all structured variables (no text yet)
    """
    np.random.seed(seed)
    print(f"Generating structured data for {n} freelancers...")
    
    # -------------------------------------------------------------------------
    # Step 1: LATENT CONFOUNDER (U) - The unobserved "ability"
    # -------------------------------------------------------------------------
    # This is the key variable that causes confounding!
    # In real data, we never observe this - but we include it for validation
    ability_score = np.random.normal(0, 1, n)  # Standard normal distribution
    print(f"  ✓ Generated latent ability scores (μ={ability_score.mean():.3f}, σ={ability_score.std():.3f})")
    
    # -------------------------------------------------------------------------
    # Step 2: DEMOGRAPHICS
    # -------------------------------------------------------------------------
    age = np.random.uniform(18, 50, n)
    
    # Experience is correlated with age but has some noise
    years_experience = np.maximum(0, (age - 18) * 0.6 + np.random.normal(0, 2, n))
    years_experience = np.minimum(years_experience, age - 18)  # Can't have more experience than age allows
    
    # Education is correlated with ability (smarter people tend to get more education)
    education_numeric = ability_score * 0.3 + np.random.normal(0, 1, n)
    education_level = pd.cut(
        education_numeric,
        bins=[-np.inf, -1.5, -0.5, 0.5, 1.5, 2.0, np.inf],
        labels=EDUCATION_LEVELS
    ).astype(str)
    
    city = np.random.choice(EGYPTIAN_CITIES, n, replace=True)
    category = np.random.choice(CATEGORIES, n, replace=True)
    print(f"  ✓ Generated demographics (age: {age.mean():.1f}±{age.std():.1f} years)")
    
    # -------------------------------------------------------------------------
    # Step 3: PLATFORM METRICS (all influenced by ability)
    # -------------------------------------------------------------------------
    # High-ability freelancers have better profiles, more skills, etc.
    profile_completeness = np.clip(50 + ability_score * 15 + np.random.normal(0, 10, n), 0, 100)
    num_skills = np.maximum(1, np.round(5 + ability_score * 2 + years_experience * 0.3 + np.random.normal(0, 2, n))).astype(int)
    portfolio_items = np.maximum(0, np.round(3 + ability_score * 1.5 + years_experience * 0.4 + np.random.normal(0, 2, n))).astype(int)
    certifications = np.maximum(0, np.round(ability_score * 0.8 + np.random.normal(0, 1, n))).astype(int)
    total_jobs = np.maximum(0, np.round(10 + years_experience * 5 + ability_score * 8 + np.random.normal(0, 10, n))).astype(int)
    success_rate = np.clip(60 + ability_score * 10 + years_experience * 1.5 + np.random.normal(0, 8, n), 0, 100)
    response_rate = np.clip(50 + ability_score * 12 + np.random.normal(0, 15, n), 0, 100)
    print(f"  ✓ Generated platform metrics (profile completeness: {profile_completeness.mean():.1f}%)")
    
    # -------------------------------------------------------------------------
    # Step 4: MARKET DEMAND
    # -------------------------------------------------------------------------
    market_demand_score = np.array([CATEGORY_DEMAND[cat] + np.random.normal(0, 0.1) for cat in category])
    market_demand_score = np.clip(market_demand_score, 0, 1)
    
    # -------------------------------------------------------------------------
    # Step 5: TREATMENT (D) - Strong selection bias based on ability
    # -------------------------------------------------------------------------
    # This is the KEY confounding mechanism!
    # High-ability freelancers are MORE likely to participate in the program
    treatment_propensity = expit(
        1.5 * ability_score +          # Strong ability effect on treatment
        0.5 * years_experience / 10 +   # Some experience effect
        0.3 * (profile_completeness - 75) / 25 +  # Profile completeness effect
        np.random.normal(0, 0.5, n)    # Random noise
    )
    program_participation = (np.random.uniform(0, 1, n) < treatment_propensity).astype(int)
    
    treatment_rate = program_participation.mean()
    print(f"  ✓ Treatment assigned (participation rate: {treatment_rate*100:.1f}%)")
    print(f"      - High ability (>1σ) participation: {program_participation[ability_score > 1].mean()*100:.1f}%")
    print(f"      - Low ability (<-1σ) participation: {program_participation[ability_score < -1].mean()*100:.1f}%")
    
    # -------------------------------------------------------------------------
    # Step 6: OUTCOME (Y) - TRUE CAUSAL EFFECT = $5.00
    # -------------------------------------------------------------------------
    category_effect = np.array([CATEGORY_EFFECTS[cat] for cat in category])
    
    hourly_earnings = (
        10 +                                        # Baseline earnings
        TRUE_CAUSAL_EFFECT * program_participation +  # *** TRUE TREATMENT EFFECT ***
        8.0 * ability_score +                       # Strong ability confounding (8x coefficient!)
        0.5 * years_experience +                    # Experience effect
        category_effect +                           # Category premium
        3.0 * market_demand_score +                 # Market demand effect
        0.05 * profile_completeness +               # Profile quality effect
        np.random.normal(0, 2, n)                   # Random noise
    )
    hourly_earnings = np.clip(hourly_earnings, 3, 80)  # Reasonable bounds
    
    # Calculate naive estimate (what we'd get without controlling for ability)
    treated_earnings = hourly_earnings[program_participation == 1].mean()
    control_earnings = hourly_earnings[program_participation == 0].mean()
    naive_effect = treated_earnings - control_earnings
    
    print(f"  ✓ Outcomes generated")
    print(f"      - Mean earnings (Treated): ${treated_earnings:.2f}")
    print(f"      - Mean earnings (Control): ${control_earnings:.2f}")
    print(f"      - Naive effect: ${naive_effect:.2f} (TRUE: ${TRUE_CAUSAL_EFFECT:.2f})")
    print(f"      - Bias: ${naive_effect - TRUE_CAUSAL_EFFECT:.2f} ({(naive_effect/TRUE_CAUSAL_EFFECT - 1)*100:+.1f}%)")
    
    # -------------------------------------------------------------------------
    # Step 7: CREATE DATAFRAME
    # -------------------------------------------------------------------------
    df = pd.DataFrame({
        'freelancer_id': [f'EGY_{i+1:05d}' for i in range(n)],
        'ability_score': ability_score,
        'age': age,
        'years_experience': years_experience,
        'education_level': education_level,
        'city': city,
        'category': category,
        'profile_completeness': profile_completeness,
        'num_skills': num_skills,
        'portfolio_items': portfolio_items,
        'certifications': certifications,
        'total_jobs': total_jobs,
        'success_rate': success_rate,
        'response_rate': response_rate,
        'market_demand_score': market_demand_score,
        'treatment_propensity': treatment_propensity,
        'program_participation': program_participation,
        'hourly_earnings': hourly_earnings
    })
    
    print(f"\n✓ Structured data generated: {df.shape[0]} rows × {df.shape[1]} columns")
    return df

In [ ]:
# =============================================================================
# FUNCTION: Call Gemini API
# =============================================================================

def call_gemini_api(prompt: str, max_retries: int = API_RETRY_ATTEMPTS) -> str:
    """
    Call the Gemini API to generate text.
    
    This function handles:
    - API request formatting
    - Retry logic with exponential backoff
    - Error handling
    
    Parameters:
    -----------
    prompt : str
        The prompt to send to Gemini
    max_retries : int
        Number of retry attempts on failure
        
    Returns:
    --------
    str or None
        Generated text, or None if all attempts fail
    """
    for attempt in range(max_retries):
        try:
            # Build the request payload
            payload = {
                "contents": [{"parts": [{"text": prompt}]}],
                "generationConfig": {
                    "temperature": 0.9,      # Higher = more creative
                    "topK": 40,              # Top-k sampling
                    "topP": 0.95,            # Nucleus sampling
                    "maxOutputTokens": 350,  # Max response length
                    "stopSequences": []
                },
                # Safety settings (allow all content for this use case)
                "safetySettings": [
                    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
                    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
                    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
                    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"}
                ]
            }
            
            # Make the API request
            response = requests.post(
                GEMINI_API_URL,
                headers={'Content-Type': 'application/json'},
                json=payload,
                timeout=30
            )
            
            # Check for API key errors
            if response.status_code == 403:
                print("\n❌ API KEY ERROR: Invalid or no access")
                print("   Check your API key at: https://aistudio.google.com/app/apikey")
                return None
            
            response.raise_for_status()
            result = response.json()
            
            # Extract the generated text
            if 'candidates' in result and len(result['candidates']) > 0:
                candidate = result['candidates'][0]
                if 'content' in candidate and 'parts' in candidate['content']:
                    return candidate['content']['parts'][0]['text'].strip()
                    
        except requests.exceptions.Timeout:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt  # Exponential backoff: 1s, 2s, 4s
                time.sleep(wait_time)
            
        except requests.exceptions.RequestException as e:
            if attempt < max_retries - 1:
                wait_time = 2 ** attempt
                time.sleep(wait_time)
                
        except Exception as e:
            if attempt < max_retries - 1:
                time.sleep(2 ** attempt)
    
    return None

In [ ]:
# =============================================================================
# FUNCTION: Generate Profile Text (Gemini API version)
# =============================================================================

def generate_profile_text_gemini(row: pd.Series) -> str:
    """
    Generate realistic profile text using Gemini API.
    
    The key insight: We create DIFFERENT prompts based on ability level!
    - HIGH ability: Professional, articulate, sophisticated vocabulary
    - MEDIUM ability: Standard, competent, clear language
    - LOW ability: Basic, simple, some grammatical issues
    
    This creates natural variation in text that reflects the latent ability,
    which our embeddings can then capture.
    
    Parameters:
    -----------
    row : pd.Series
        A row from our DataFrame with freelancer info
        
    Returns:
    --------
    str
        Generated profile text
    """
    ability = row['ability_score']
    category = row['category']
    experience = row['years_experience']
    city = row['city']
    education = row['education_level']
    num_skills = row['num_skills']
    market_demand = row['market_demand_score']
    
    # ----- HIGH ABILITY PROMPT (ability > 1.0) -----
    if ability > 1.0:
        prompt = f"""Write a highly professional, articulate, and persuasive Upwork freelancer profile summary for an expert Egyptian professional.

Category: {category}
Location: {city}, Egypt
Experience: {experience:.1f} years
Education: {education}
Number of skills: {num_skills}

Requirements:
- Use perfect grammar and sophisticated vocabulary
- Highlight expertise and unique value proposition
- Mention specific technical skills and relevant tools
- Include client-focused language (ROI, results, quality, impact)
- {"Emphasize high-demand skills and cutting-edge expertise" if market_demand > 0.7 else "Focus on proven track record and reliability"}
- Professional tone, confident but not arrogant
- 100-150 words
- Sound like a top 1% freelancer on Upwork
- Write in FIRST PERSON ("I am...", "I specialize...", etc.)
- DO NOT use placeholder names or brackets
- Be specific and concrete

Write only the profile summary, nothing else."""
    
    # ----- LOW ABILITY PROMPT (ability < -1.0) -----
    elif ability < -1.0:
        prompt = f"""Write a short, basic Upwork freelancer profile for an entry-level Egyptian professional who is still developing their skills.

Category: {category}
Location: {city}, Egypt
Experience: {experience:.1f} years

Requirements:
- Simple language, generic phrasing
- Include 1-2 minor grammatical imperfections (lowercase "i", missing punctuation, casual phrasing)
- Brief and to the point (60-90 words)
- Less polished, more straightforward
- Use common clichés like "hard working", "dedicated", "passionate"
- Sound like a beginner trying their best
- Write in FIRST PERSON ("I am...", "i have...", etc.)
- DO NOT use placeholder names or brackets
- Keep it authentic and simple

Write only the profile summary, nothing else."""
    
    # ----- MEDIUM ABILITY PROMPT (-1.0 <= ability <= 1.0) -----
    else:
        prompt = f"""Write a standard, competent Upwork freelancer profile summary for an Egyptian professional.

Category: {category}
Location: {city}, Egypt
Experience: {experience:.1f} years
Education: {education}

Requirements:
- Clear and professional language
- Mention relevant skills and experience
- Standard grammar (no major errors)
- 80-120 words
- Sound like an average, reliable freelancer
- Professional but not exceptional
- Write in FIRST PERSON ("I am...", "I offer...", etc.)
- DO NOT use placeholder names or brackets
- Be straightforward and honest

Write only the profile summary, nothing else."""
    
    # Call the API
    return call_gemini_api(prompt)

In [ ]:
# =============================================================================
# FALLBACK: Template-Based Profile Generation (no API needed)
# =============================================================================

# Templates for different ability levels
HIGH_ABILITY_TEMPLATES = [
    "Seasoned {category} specialist with {exp} years of proven expertise. Demonstrated excellence in delivering exceptional results for clients across {city}'s competitive market. I bring deep proficiency in industry best practices and cutting-edge methodologies.",
    "Expert {category} professional leveraging {exp} years of industry experience. My track record speaks for itself - consistently delivering high-impact solutions that drive measurable business outcomes. Based in {city}, serving global clientele with distinction.",
    "Distinguished {category} consultant with {exp} years of specialized experience. I combine technical mastery with strategic thinking to deliver innovative, high-quality solutions. {city}-based, internationally recognized for excellence.",
]

MED_ABILITY_TEMPLATES = [
    "Experienced {category} professional with {exp} years in the field. Skilled in delivering quality work and meeting deadlines. Based in {city}, I'm ready to help with your projects.",
    "{category} specialist with {exp} years of solid experience. I provide reliable services and maintain good communication with clients. Located in {city}, Egypt.",
    "Professional {category} provider with {exp} years of background. Good knowledge of industry standards and practices. Working from {city} to serve your needs.",
]

LOW_ABILITY_TEMPLATES = [
    "I do {category} work with {exp} years experience. i can help you with your projects. Im in {city} and ready to work hard.",
    "{category} freelancer here with {exp} years. I work hard and am very dedicated to my clients satisfaction. From {city}.",
    "Hello! {exp} years doing {category}. I am passionate about this work and will do my best for you. Based {city}, Egypt.",
]

def generate_profile_text_template(row: pd.Series) -> str:
    """
    Generate profile text using templates (fallback when API unavailable).
    
    This is faster but produces less realistic/varied text than Gemini.
    """
    ability = row['ability_score']
    category = row['category']
    experience = int(row['years_experience'])
    city = row['city']
    
    if ability > 1.0:
        template = np.random.choice(HIGH_ABILITY_TEMPLATES)
    elif ability > -1.0:
        template = np.random.choice(MED_ABILITY_TEMPLATES)
    else:
        template = np.random.choice(LOW_ABILITY_TEMPLATES)
    
    return template.format(category=category, exp=experience, city=city)

print("✓ Text generation functions defined")

In [ ]:
# =============================================================================
# FUNCTION: Add Profile Text to DataFrame
# =============================================================================

def add_profile_texts(df: pd.DataFrame, use_gemini: bool = USE_GEMINI_API) -> pd.DataFrame:
    """
    Add profile text to each freelancer in the DataFrame.
    
    Parameters:
    -----------
    df : pd.DataFrame
        DataFrame with structured data
    use_gemini : bool
        If True, use Gemini API; otherwise use templates
        
    Returns:
    --------
    pd.DataFrame
        DataFrame with added 'profile_text' column
    """
    print("\n" + "="*70)
    if use_gemini:
        print("GENERATING PROFILE TEXT USING GEMINI API")
        print("="*70)
        print(f"Model: Gemini 1.5 Flash")
        print(f"Total profiles: {len(df)}")
        print(f"Estimated time: {len(df) * 2 / 60:.1f} minutes")
    else:
        print("GENERATING PROFILE TEXT USING TEMPLATES")
        print("="*70)
        print(f"Total profiles: {len(df)}")
        print("Note: Templates are faster but less realistic than Gemini API")
    print("="*70 + "\n")
    
    profile_texts = []
    failed_count = 0
    
    for idx in tqdm(range(len(df)), desc="Generating profiles"):
        row = df.iloc[idx]
        
        if use_gemini:
            text = generate_profile_text_gemini(row)
            if text is None:
                # Fallback to template if API fails
                text = generate_profile_text_template(row)
                failed_count += 1
        else:
            text = generate_profile_text_template(row)
        
        profile_texts.append(text)
        
        # Rate limiting for API calls
        if use_gemini and (idx + 1) % API_BATCH_SIZE == 0 and idx < len(df) - 1:
            time.sleep(2)  # Pause to avoid rate limiting
    
    df = df.copy()
    df['profile_text'] = profile_texts
    
    print(f"\n✓ Generated {len(profile_texts)} profiles")
    if use_gemini and failed_count > 0:
        print(f"  - API successes: {len(df) - failed_count}")
        print(f"  - Fallback to template: {failed_count}")
    
    # Show sample profiles
    print("\n" + "-"*70)
    print("SAMPLE PROFILES BY ABILITY TIER")
    print("-"*70)
    
    # High ability example
    high_df = df[df.ability_score > 1.5]
    if len(high_df) > 0:
        print("\n[HIGH ABILITY - Score > 1.5]")
        sample = high_df.iloc[0]
        print(f"Ability: {sample.ability_score:.2f} | Category: {sample.category}")
        print(f"Text: {sample.profile_text[:200]}...\n")
    
    # Medium ability example
    med_df = df[(df.ability_score > -0.5) & (df.ability_score < 0.5)]
    if len(med_df) > 0:
        print("[MEDIUM ABILITY - Score ≈ 0]")
        sample = med_df.iloc[0]
        print(f"Ability: {sample.ability_score:.2f} | Category: {sample.category}")
        print(f"Text: {sample.profile_text[:200]}...\n")
    
    # Low ability example
    low_df = df[df.ability_score < -1.5]
    if len(low_df) > 0:
        print("[LOW ABILITY - Score < -1.5]")
        sample = low_df.iloc[0]
        print(f"Ability: {sample.ability_score:.2f} | Category: {sample.category}")
        print(f"Text: {sample.profile_text[:200]}...\n")
    
    return df

In [ ]:
# =============================================================================
# EXECUTE: Generate the Full Dataset
# =============================================================================

print("\n" + "="*70)
print("STARTING DATA GENERATION PIPELINE")
print("="*70)
print(f"\nConfiguration:")
print(f"  - Sample size: {N_SAMPLES}")
print(f"  - Use Gemini API: {USE_GEMINI_API}")
print(f"  - Random seed: {RANDOM_SEED}")
print(f"  - True causal effect: ${TRUE_CAUSAL_EFFECT:.2f}")
print("\n")

# Step 1: Generate structured data
df = generate_structured_data(N_SAMPLES, seed=RANDOM_SEED)

# Step 2: Add profile texts
df = add_profile_texts(df, use_gemini=USE_GEMINI_API)

# Step 3: Save the dataset
df.to_parquet(DATA_OUTPUT_PATH, index=False)
print(f"\n💾 Dataset saved to: {DATA_OUTPUT_PATH}")
print(f"   Shape: {df.shape}")

print("\n" + "="*70)
print("✓ DATA GENERATION COMPLETE!")
print("="*70)

---

## SECTION 3: Exploratory Data Analysis (EDA)

Before applying causal methods, let's visualize the confounding problem.

In [ ]:
# =============================================================================
# EXPLORATORY DATA ANALYSIS
# =============================================================================

fig, axes = plt.subplots(2, 3, figsize=(18, 12))

# --- Plot 1: Ability Distribution by Treatment ---
# This shows the SELECTION BIAS: high-ability people are more likely to be treated
axes[0, 0].hist(df[df['program_participation']==0]['ability_score'], 
                bins=50, alpha=0.6, label='Control', density=True, color='red')
axes[0, 0].hist(df[df['program_participation']==1]['ability_score'], 
                bins=50, alpha=0.6, label='Treated', density=True, color='blue')
axes[0, 0].set_xlabel('Ability Score (Unobserved)', fontsize=12)
axes[0, 0].set_ylabel('Density', fontsize=12)
axes[0, 0].set_title('Selection Bias: High-Ability → Treatment', fontsize=14, fontweight='bold')
axes[0, 0].legend()
axes[0, 0].axvline(0, color='black', linestyle='--', alpha=0.3)

# --- Plot 2: Earnings vs Ability ---
# This shows the OUTCOME CONFOUNDING: ability affects earnings
for treatment in [0, 1]:
    mask = df['program_participation'] == treatment
    axes[0, 1].scatter(df[mask]['ability_score'], df[mask]['hourly_earnings'],
                      alpha=0.3, s=10, label=f"{'Treated' if treatment else 'Control'}")
axes[0, 1].set_xlabel('Ability Score', fontsize=12)
axes[0, 1].set_ylabel('Hourly Earnings ($)', fontsize=12)
axes[0, 1].set_title('Outcome Confounding: Ability → Earnings', fontsize=14, fontweight='bold')
axes[0, 1].legend()

# --- Plot 3: Treatment Propensity by Ability ---
# Shows P(Treatment | Ability)
ability_bins = pd.cut(df['ability_score'], bins=20)
propensity_by_ability = df.groupby(ability_bins, observed=True)['program_participation'].mean()
bin_centers = [interval.mid for interval in propensity_by_ability.index]
axes[0, 2].plot(bin_centers, propensity_by_ability.values, 'o-', linewidth=2, markersize=6, color='purple')
axes[0, 2].set_xlabel('Ability Score', fontsize=12)
axes[0, 2].set_ylabel('P(Treatment)', fontsize=12)
axes[0, 2].set_title('Treatment Propensity by Ability', fontsize=14, fontweight='bold')
axes[0, 2].grid(True, alpha=0.3)
axes[0, 2].axhline(0.5, color='red', linestyle='--', alpha=0.5)

# --- Plot 4: Naive vs True Effect ---
treated_mean = df[df['program_participation']==1]['hourly_earnings'].mean()
control_mean = df[df['program_participation']==0]['hourly_earnings'].mean()
naive_ate = treated_mean - control_mean

estimates = ['True Effect', 'Naive Estimate']
values = [TRUE_CAUSAL_EFFECT, naive_ate]
colors = ['green', 'red']
bars = axes[1, 0].bar(estimates, values, color=colors, alpha=0.7, edgecolor='black')
axes[1, 0].axhline(TRUE_CAUSAL_EFFECT, color='green', linestyle='--', linewidth=2)
axes[1, 0].set_ylabel('Treatment Effect ($)', fontsize=12)
axes[1, 0].set_title(f'Naive Bias: {(naive_ate/TRUE_CAUSAL_EFFECT - 1)*100:+.1f}%', 
                     fontsize=14, fontweight='bold', color='red')
for bar, val in zip(bars, values):
    axes[1, 0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                    f'${val:.2f}', ha='center', fontsize=11, fontweight='bold')

# --- Plot 5: Observable vs Ability Correlations ---
observables = ['Age', 'Experience', 'Completeness', 'Skills', 'Portfolio']
observable_vars = ['age', 'years_experience', 'profile_completeness', 'num_skills', 'portfolio_items']
correlations = [df[var].corr(df['ability_score']) for var in observable_vars]
axes[1, 1].barh(observables, correlations, color='steelblue', alpha=0.7, edgecolor='black')
axes[1, 1].set_xlabel('Correlation with Ability', fontsize=12)
axes[1, 1].set_title('Observables vs Ability', fontsize=14, fontweight='bold')
axes[1, 1].axvline(0, color='black', linewidth=0.8)
axes[1, 1].grid(True, alpha=0.3, axis='x')

# --- Plot 6: Category Distribution ---
category_counts = df['category'].value_counts()
axes[1, 2].barh(category_counts.index, category_counts.values, color='coral', alpha=0.7, edgecolor='black')
axes[1, 2].set_xlabel('Count', fontsize=12)
axes[1, 2].set_title('Distribution by Category', fontsize=14, fontweight='bold')
axes[1, 2].grid(True, alpha=0.3, axis='x')

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/eda_confounding_analysis.png', dpi=150, bbox_inches='tight')
plt.show()

# Print summary
print("\n" + "="*70)
print("KEY FINDINGS FROM EDA")
print("="*70)
print(f"True causal effect:        ${TRUE_CAUSAL_EFFECT:.2f}")
print(f"Naive estimate:            ${naive_ate:.2f}")
print(f"Bias:                      ${naive_ate - TRUE_CAUSAL_EFFECT:.2f} ({(naive_ate/TRUE_CAUSAL_EFFECT - 1)*100:+.1f}%)")
print(f"\n→ The naive estimate is SEVERELY BIASED due to confounding!")
print(f"→ We need to use text embeddings to proxy for unobserved ability.")
print("="*70)

---

## SECTION 4: Text Embeddings

### What Are Embeddings?

Embeddings convert text into numerical vectors that capture semantic meaning:
- Similar texts → similar vectors
- Different texts → different vectors

### Why This Works for Causal Inference

High-ability freelancers write BETTER profiles (more articulate, professional, sophisticated vocabulary). The embeddings capture this variation, allowing us to **proxy for the unobserved ability confounder**.

### Process

1. Load pre-trained SentenceTransformer model
2. Generate 384-dimensional embeddings for each profile
3. Standardize embeddings (mean=0, std=1)
4. Apply PCA for dimensionality reduction

In [ ]:
# =============================================================================
# GENERATE TEXT EMBEDDINGS
# =============================================================================

print("\n" + "="*70)
print("GENERATING TEXT EMBEDDINGS")
print("="*70)

# Load the pre-trained embedding model
print(f"\nLoading SentenceTransformer model: {EMBEDDING_MODEL}...")
embedding_model = SentenceTransformer(EMBEDDING_MODEL)
embedding_dim = embedding_model.get_sentence_embedding_dimension()
print(f"✓ Model loaded (embedding dimension: {embedding_dim})")

# Generate embeddings
print(f"\nGenerating embeddings for {len(df)} profiles...")
embeddings_raw = embedding_model.encode(
    df['profile_text'].tolist(),
    show_progress_bar=True,
    batch_size=64,
    convert_to_numpy=True
)
print(f"✓ Generated embeddings: {embeddings_raw.shape}")

# Standardize embeddings
print("\nStandardizing embeddings...")
scaler = StandardScaler()
embeddings_scaled = scaler.fit_transform(embeddings_raw)
print(f"✓ Standardized (mean={embeddings_scaled.mean():.6f}, std={embeddings_scaled.std():.4f})")

# Save embeddings
np.save(f'{PROJECT_DIR}/embeddings_raw.npy', embeddings_raw)
np.save(f'{PROJECT_DIR}/embeddings_scaled.npy', embeddings_scaled)
joblib.dump(scaler, f'{PROJECT_DIR}/scaler.pkl')
print(f"✓ Saved embeddings and scaler to disk")

In [ ]:
# =============================================================================
# APPLY PCA FOR DIMENSIONALITY REDUCTION
# =============================================================================

print("\n" + "="*70)
print("APPLYING PCA DIMENSIONALITY REDUCTION")
print("="*70)

# Fit PCA on all components first to analyze variance
pca_full = PCA()
pca_full.fit(embeddings_scaled)

# Calculate cumulative variance
cumulative_variance = np.cumsum(pca_full.explained_variance_ratio_)
n_components_90 = np.argmax(cumulative_variance >= 0.90) + 1
n_components_95 = np.argmax(cumulative_variance >= 0.95) + 1

print(f"\nVariance Analysis:")
print(f"  - Components for 90% variance: {n_components_90}")
print(f"  - Components for 95% variance: {n_components_95}")
print(f"  - Using {MAX_PCA_COMPONENTS} components for analysis")

# Apply final PCA
pca = PCA(n_components=MAX_PCA_COMPONENTS)
pca_embeddings = pca.fit_transform(embeddings_scaled)
print(f"\n✓ PCA applied: {embeddings_scaled.shape} → {pca_embeddings.shape}")
print(f"  - Variance explained: {pca.explained_variance_ratio_.sum()*100:.1f}%")

# Add PCA components to dataframe
for i in range(MAX_PCA_COMPONENTS):
    df[f'pca_{i+1}'] = pca_embeddings[:, i]

# Save PCA model
joblib.dump(pca, f'{PROJECT_DIR}/pca_model.pkl')
print(f"✓ Added pca_1 through pca_{MAX_PCA_COMPONENTS} to dataframe")

In [ ]:
# =============================================================================
# VALIDATE EMBEDDINGS: Correlation with Ability
# =============================================================================

print("\n" + "="*70)
print("VALIDATING EMBEDDING QUALITY")
print("="*70)

# Calculate correlations between PCA components and ability
pca_correlations = [df[f'pca_{i}'].corr(df['ability_score']) for i in range(1, 21)]

# Visualization
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Plot 1: Variance explained
axes[0].plot(range(1, MAX_PCA_COMPONENTS+1), cumulative_variance[:MAX_PCA_COMPONENTS], 'o-', linewidth=2, markersize=4)
axes[0].axhline(0.90, color='red', linestyle='--', alpha=0.7, label='90% threshold')
axes[0].axhline(0.95, color='orange', linestyle='--', alpha=0.7, label='95% threshold')
axes[0].axvline(n_components_90, color='red', linestyle=':', alpha=0.7)
axes[0].set_xlabel('Number of Components', fontsize=12)
axes[0].set_ylabel('Cumulative Variance Explained', fontsize=12)
axes[0].set_title('PCA Variance Explained', fontsize=14, fontweight='bold')
axes[0].grid(True, alpha=0.3)
axes[0].legend()

# Plot 2: Correlations with ability
colors = ['green' if abs(c) > 0.3 else 'steelblue' for c in pca_correlations]
axes[1].bar(range(1, 21), pca_correlations, color=colors, alpha=0.7, edgecolor='black')
axes[1].set_xlabel('PCA Component', fontsize=12)
axes[1].set_ylabel('Correlation with Ability', fontsize=12)
axes[1].set_title('PCA Components Capture Ability Signal', fontsize=14, fontweight='bold')
axes[1].axhline(0, color='black', linewidth=0.8)
axes[1].axhline(0.3, color='red', linestyle='--', alpha=0.5, label='Strong correlation')
axes[1].axhline(-0.3, color='red', linestyle='--', alpha=0.5)
axes[1].grid(True, alpha=0.3, axis='y')
axes[1].legend()

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/pca_validation.png', dpi=150, bbox_inches='tight')
plt.show()

# Print validation results
print(f"\nTop 5 PCA Components vs Ability:")
for i in range(5):
    strength = "STRONG" if abs(pca_correlations[i]) > 0.3 else "moderate" if abs(pca_correlations[i]) > 0.1 else "weak"
    print(f"  PC{i+1}: r = {pca_correlations[i]:+.4f} ({strength})")

max_corr = max(abs(c) for c in pca_correlations)
print(f"\n→ Maximum |correlation| with ability: {max_corr:.4f}")
if max_corr > 0.3:
    print("→ GOOD: Text embeddings strongly capture latent ability!")
elif max_corr > 0.1:
    print("→ OK: Text embeddings moderately capture latent ability.")
else:
    print("→ WARNING: Weak correlation - embeddings may not capture ability well.")

print("="*70)

In [ ]:
# =============================================================================
# SAVE ENHANCED DATASET
# =============================================================================

# Save the dataset with embeddings
df.to_parquet(EMBEDDINGS_OUTPUT_PATH, index=False)
print(f"\n✓ Enhanced dataset saved to: {EMBEDDINGS_OUTPUT_PATH}")
print(f"  Shape: {df.shape}")
print(f"  Original columns: 18")
print(f"  Added PCA columns: {MAX_PCA_COMPONENTS}")
print(f"  Added profile_text column: 1")

---

## SECTION 5: Causal Estimation Methods

Now we apply multiple causal inference methods to estimate the treatment effect.

### Method Overview

| Method | Description | Expected Performance |
|--------|-------------|---------------------|
| **Naive OLS** | Simple regression without confounding control | SEVERELY BIASED |
| **DML (High-Dim)** | Double ML with 384D embeddings + LassoCV | Good |
| **DML (PCA+RF)** | Double ML with PCA components + Random Forest | Good |
| **Bayesian** | Full posterior inference with Bambi | Good + Uncertainty |

### Double Machine Learning (DML)

DML uses a two-stage residualization approach:

1. **Residualize Y**: Remove the effect of confounders on outcome
2. **Residualize D**: Remove the effect of confounders on treatment  
3. **Estimate τ**: Regress residualized outcome on residualized treatment

This achieves **Neyman orthogonality**, making the estimate robust to nuisance parameter estimation errors.

In [ ]:
# =============================================================================
# PREPARE FEATURE SETS FOR CAUSAL ESTIMATION
# =============================================================================

print("\n" + "="*70)
print("PREPARING DATA FOR CAUSAL ESTIMATION")
print("="*70)

# Define feature sets
basic_features = ['age', 'years_experience', 'profile_completeness']
pca_features_dml = [f'pca_{i}' for i in range(1, DML_N_PCA_COMPONENTS + 1)]
pca_features_bayes = [f'pca_{i}' for i in range(1, BAYES_N_PCA_COMPONENTS + 1)]

# Extract arrays
Y = df['hourly_earnings'].values           # Outcome
D = df['program_participation'].values     # Treatment
X_basic = df[basic_features].values        # Basic observables
X_embeddings = embeddings_scaled           # Full 384D embeddings
X_pca_dml = df[pca_features_dml].values   # PCA for DML
X_pca_bayes = df[pca_features_bayes].values  # PCA for Bayesian

print(f"\nFeature sets:")
print(f"  - Basic observables: {X_basic.shape} {basic_features}")
print(f"  - Full embeddings: {X_embeddings.shape}")
print(f"  - PCA for DML: {X_pca_dml.shape}")
print(f"  - PCA for Bayesian: {X_pca_bayes.shape}")

# Storage for results
results = {}

print("\n" + "="*70)

In [ ]:
# =============================================================================
# METHOD 1: NAIVE OLS (BASELINE - EXPECTED TO BE BIASED)
# =============================================================================

print("\n" + "="*70)
print("METHOD 1: Naive OLS Regression (Baseline)")
print("="*70)
print("\nSpecification: Y ~ D + age + experience + profile_completeness")
print("NOTE: This does NOT control for ability → Expected to be SEVERELY BIASED\n")

# Build the design matrix: [intercept, treatment, basic_features]
X_naive = np.column_stack([np.ones(len(Y)), D, X_basic])

# OLS estimation
beta_naive = np.linalg.lstsq(X_naive, Y, rcond=None)[0]
Y_pred_naive = X_naive @ beta_naive
residuals_naive = Y - Y_pred_naive

# Standard errors
n = len(Y)
k = X_naive.shape[1]
mse = np.sum(residuals_naive**2) / (n - k)
var_beta = mse * np.linalg.inv(X_naive.T @ X_naive)
se_naive = np.sqrt(np.diag(var_beta))

# Treatment effect (coefficient on D, which is at index 1)
tau_naive = beta_naive[1]
se_tau_naive = se_naive[1]
ci_naive = (tau_naive - 1.96 * se_tau_naive, tau_naive + 1.96 * se_tau_naive)

# Store results
results['Naive OLS'] = {
    'estimate': tau_naive,
    'se': se_tau_naive,
    'ci_lower': ci_naive[0],
    'ci_upper': ci_naive[1],
    'bias': tau_naive - TRUE_CAUSAL_EFFECT
}

# Print results
print(f"✓ Naive OLS Estimate: ${tau_naive:.2f}")
print(f"  Standard Error: ${se_tau_naive:.2f}")
print(f"  95% CI: [${ci_naive[0]:.2f}, ${ci_naive[1]:.2f}]")
print(f"\n  True Effect: ${TRUE_CAUSAL_EFFECT:.2f}")
print(f"  Bias: ${tau_naive - TRUE_CAUSAL_EFFECT:.2f} ({(tau_naive/TRUE_CAUSAL_EFFECT - 1)*100:+.1f}% error)")
print(f"\n→ As expected, naive OLS is SEVERELY BIASED due to omitted ability!")

In [ ]:
# =============================================================================
# METHOD 2A: DOUBLE MACHINE LEARNING (High-Dimensional Embeddings)
# =============================================================================

print("\n" + "="*70)
print("METHOD 2A: Double Machine Learning - High-Dimensional Embeddings")
print("="*70)
print("\nStrategy: Use all 384 embedding dimensions with LassoCV")
print("Nuisance models: LassoCV (L1 regularization handles high dimensionality)\n")

# Combine basic features with embeddings
X_full_emb = np.column_stack([X_basic, X_embeddings])
print(f"Control variables: {X_full_emb.shape[1]} (3 basic + 384 embeddings)")

# --- Step 1: Residualize Y on X ---
# This removes the effect of confounders on the outcome
print("\n[Step 1/3] Residualizing outcome Y on confounders...")
lasso_y = LassoCV(cv=DML_CV_FOLDS, random_state=RANDOM_SEED, max_iter=10000)
Y_pred = cross_val_predict(lasso_y, X_full_emb, Y, cv=DML_CV_FOLDS)
Y_res = Y - Y_pred
print(f"✓ Y residuals: mean={Y_res.mean():.6f}, std={Y_res.std():.3f}")

# --- Step 2: Residualize D on X ---
# This removes the effect of confounders on the treatment
print("\n[Step 2/3] Residualizing treatment D on confounders...")
lasso_d = LassoCV(cv=DML_CV_FOLDS, random_state=RANDOM_SEED, max_iter=10000)
D_pred = cross_val_predict(lasso_d, X_full_emb, D, cv=DML_CV_FOLDS)
D_res = D - D_pred
print(f"✓ D residuals: mean={D_res.mean():.6f}, std={D_res.std():.3f}")

# --- Step 3: Estimate treatment effect ---
# This is the Neyman orthogonal moment: τ = E[D_res * Y_res] / E[D_res^2]
print("\n[Step 3/3] Estimating causal effect via Neyman orthogonality...")
tau_dml_emb = np.sum(D_res * Y_res) / np.sum(D_res ** 2)

# Standard error (robust to nuisance parameter estimation)
residuals_final = Y_res - tau_dml_emb * D_res
se_dml_emb = np.sqrt(np.mean(residuals_final**2) / (np.mean(D_res**2) * n))
ci_dml_emb = (tau_dml_emb - 1.96 * se_dml_emb, tau_dml_emb + 1.96 * se_dml_emb)

# Store results
results['DML (High-Dim)'] = {
    'estimate': tau_dml_emb,
    'se': se_dml_emb,
    'ci_lower': ci_dml_emb[0],
    'ci_upper': ci_dml_emb[1],
    'bias': tau_dml_emb - TRUE_CAUSAL_EFFECT
}

# Print results
print(f"\n✓ DML (High-Dim) Estimate: ${tau_dml_emb:.2f}")
print(f"  Standard Error: ${se_dml_emb:.2f}")
print(f"  95% CI: [${ci_dml_emb[0]:.2f}, ${ci_dml_emb[1]:.2f}]")
print(f"\n  True Effect: ${TRUE_CAUSAL_EFFECT:.2f}")
print(f"  Bias: ${tau_dml_emb - TRUE_CAUSAL_EFFECT:.2f} ({(tau_dml_emb/TRUE_CAUSAL_EFFECT - 1)*100:+.1f}% error)")
in_ci = ci_dml_emb[0] <= TRUE_CAUSAL_EFFECT <= ci_dml_emb[1]
print(f"  True effect in CI: {'YES ✓' if in_ci else 'NO ✗'}")

In [ ]:
# =============================================================================
# METHOD 2B: DOUBLE MACHINE LEARNING (PCA + Random Forest)
# =============================================================================

print("\n" + "="*70)
print("METHOD 2B: Double Machine Learning - PCA + Random Forest")
print("="*70)
print(f"\nStrategy: Use top {DML_N_PCA_COMPONENTS} PCA components with Random Forest")
print("Nuisance models: RandomForestRegressor (handles non-linearities)\n")

# Combine basic features with PCA
X_full_pca = np.column_stack([X_basic, X_pca_dml])
print(f"Control variables: {X_full_pca.shape[1]} (3 basic + {DML_N_PCA_COMPONENTS} PCA)")

# --- Step 1: Residualize Y ---
print("\n[Step 1/3] Residualizing outcome Y...")
rf_y = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1)
Y_pred_rf = cross_val_predict(rf_y, X_full_pca, Y, cv=DML_CV_FOLDS)
Y_res_rf = Y - Y_pred_rf
print(f"✓ Y residuals: mean={Y_res_rf.mean():.6f}, std={Y_res_rf.std():.3f}")

# --- Step 2: Residualize D ---
print("\n[Step 2/3] Residualizing treatment D...")
rf_d = RandomForestRegressor(n_estimators=100, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1)
D_pred_rf = cross_val_predict(rf_d, X_full_pca, D, cv=DML_CV_FOLDS)
D_res_rf = D - D_pred_rf
print(f"✓ D residuals: mean={D_res_rf.mean():.6f}, std={D_res_rf.std():.3f}")

# --- Step 3: Estimate treatment effect ---
print("\n[Step 3/3] Estimating causal effect...")
tau_dml_rf = np.sum(D_res_rf * Y_res_rf) / np.sum(D_res_rf ** 2)

# Standard error
residuals_final_rf = Y_res_rf - tau_dml_rf * D_res_rf
se_dml_rf = np.sqrt(np.mean(residuals_final_rf**2) / (np.mean(D_res_rf**2) * n))
ci_dml_rf = (tau_dml_rf - 1.96 * se_dml_rf, tau_dml_rf + 1.96 * se_dml_rf)

# Store results
results['DML (PCA+RF)'] = {
    'estimate': tau_dml_rf,
    'se': se_dml_rf,
    'ci_lower': ci_dml_rf[0],
    'ci_upper': ci_dml_rf[1],
    'bias': tau_dml_rf - TRUE_CAUSAL_EFFECT
}

# Print results
print(f"\n✓ DML (PCA+RF) Estimate: ${tau_dml_rf:.2f}")
print(f"  Standard Error: ${se_dml_rf:.2f}")
print(f"  95% CI: [${ci_dml_rf[0]:.2f}, ${ci_dml_rf[1]:.2f}]")
print(f"\n  True Effect: ${TRUE_CAUSAL_EFFECT:.2f}")
print(f"  Bias: ${tau_dml_rf - TRUE_CAUSAL_EFFECT:.2f} ({(tau_dml_rf/TRUE_CAUSAL_EFFECT - 1)*100:+.1f}% error)")
in_ci = ci_dml_rf[0] <= TRUE_CAUSAL_EFFECT <= ci_dml_rf[1]
print(f"  True effect in CI: {'YES ✓' if in_ci else 'NO ✗'}")

---

## SECTION 6: Bayesian Inference with Bambi

### Why Bayesian?

Bayesian methods provide:
- **Full posterior distribution**: Not just a point estimate, but entire uncertainty
- **Credible intervals**: "There is 94% probability the effect is in this range"
- **Natural uncertainty quantification**: Essential for policy decisions

### Bambi

Bambi is a Bayesian model-building interface built on PyMC. It uses R-style formulas:

```
hourly_earnings ~ program_participation + age + experience + completeness + pca_1 + ... + pca_10
```

In [ ]:
# =============================================================================
# METHOD 3: BAYESIAN INFERENCE WITH BAMBI
# =============================================================================

print("\n" + "="*70)
print("METHOD 3: Bayesian Inference with Bambi/PyMC")
print("="*70)
print(f"\nStrategy: Use top {BAYES_N_PCA_COMPONENTS} PCA components with weakly informative priors")
print(f"MCMC: {MCMC_DRAWS} draws, {MCMC_TUNE} tuning steps")
print("\nNote: This may take a few minutes...\n")

# Prepare data for Bambi
df_bayes = df[['hourly_earnings', 'program_participation'] + 
              basic_features + pca_features_bayes].copy()

# Build model formula
formula = 'hourly_earnings ~ program_participation + age + years_experience + profile_completeness'
for i in range(1, BAYES_N_PCA_COMPONENTS + 1):
    formula += f' + pca_{i}'

print(f"Model formula:\n{formula}\n")

# Build and fit the model
print("Building Bambi model...")
bayes_model = bmb.Model(formula, df_bayes)
print("✓ Model built\n")

print("Running MCMC sampling...")
idata = bayes_model.fit(
    draws=MCMC_DRAWS,
    tune=MCMC_TUNE,
    random_seed=RANDOM_SEED,
    progressbar=True
)
print("\n✓ MCMC sampling complete!")

In [ ]:
# =============================================================================
# ANALYZE BAYESIAN POSTERIOR
# =============================================================================

print("\n" + "="*70)
print("BAYESIAN POSTERIOR ANALYSIS")
print("="*70)

# Extract posterior samples for treatment effect
posterior_samples = idata.posterior['program_participation'].values.flatten()
tau_bayes = np.mean(posterior_samples)
se_bayes = np.std(posterior_samples)

# Compute HDI (Highest Density Interval)
hdi = az.hdi(idata, hdi_prob=HDI_PROB)
hdi_lower = float(hdi['program_participation'].values[0])
hdi_upper = float(hdi['program_participation'].values[1])

# Probabilistic statements
prob_positive = np.mean(posterior_samples > 0)
prob_gt_true = np.mean(posterior_samples > TRUE_CAUSAL_EFFECT)

# Store results
results['Bayesian'] = {
    'estimate': tau_bayes,
    'se': se_bayes,
    'ci_lower': hdi_lower,
    'ci_upper': hdi_upper,
    'bias': tau_bayes - TRUE_CAUSAL_EFFECT
}

# Print results
print(f"\n✓ Bayesian Estimate (Posterior Mean): ${tau_bayes:.2f}")
print(f"  Posterior SD: ${se_bayes:.2f}")
print(f"  {int(HDI_PROB*100)}% HDI: [${hdi_lower:.2f}, ${hdi_upper:.2f}]")
print(f"\n  True Effect: ${TRUE_CAUSAL_EFFECT:.2f}")
print(f"  Bias: ${tau_bayes - TRUE_CAUSAL_EFFECT:.2f} ({(tau_bayes/TRUE_CAUSAL_EFFECT - 1)*100:+.1f}% error)")
in_hdi = hdi_lower <= TRUE_CAUSAL_EFFECT <= hdi_upper
print(f"  True effect in HDI: {'YES ✓' if in_hdi else 'NO ✗'}")
print(f"\nProbabilistic statements:")
print(f"  P(effect > $0): {prob_positive*100:.1f}%")
print(f"  P(effect > ${TRUE_CAUSAL_EFFECT:.2f}): {prob_gt_true*100:.1f}%")

In [ ]:
# =============================================================================
# VISUALIZE BAYESIAN POSTERIOR
# =============================================================================

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# --- Plot 1: Posterior Distribution ---
axes[0].hist(posterior_samples, bins=50, density=True, alpha=0.7, 
             color='steelblue', edgecolor='black', label='Posterior')
axes[0].axvline(tau_bayes, color='blue', linestyle='-', linewidth=2.5, 
                label=f'Posterior Mean: ${tau_bayes:.2f}')
axes[0].axvline(TRUE_CAUSAL_EFFECT, color='green', linestyle='--', linewidth=2.5,
                label=f'True Effect: ${TRUE_CAUSAL_EFFECT:.2f}')
axes[0].axvline(hdi_lower, color='red', linestyle=':', linewidth=2, alpha=0.7)
axes[0].axvline(hdi_upper, color='red', linestyle=':', linewidth=2, alpha=0.7,
                label=f'{int(HDI_PROB*100)}% HDI: [${hdi_lower:.2f}, ${hdi_upper:.2f}]')
axes[0].fill_betweenx([0, axes[0].get_ylim()[1] if axes[0].get_ylim()[1] > 0 else 1], 
                       hdi_lower, hdi_upper, color='red', alpha=0.1)
axes[0].set_xlabel('Treatment Effect ($)', fontsize=12)
axes[0].set_ylabel('Density', fontsize=12)
axes[0].set_title('Bayesian Posterior Distribution', fontsize=14, fontweight='bold')
axes[0].legend(fontsize=10)
axes[0].grid(True, alpha=0.3)

# --- Plot 2: MCMC Trace ---
trace = idata.posterior['program_participation'].values[0, :]  # First chain
axes[1].plot(trace, linewidth=0.8, alpha=0.7, color='steelblue')
axes[1].axhline(tau_bayes, color='blue', linestyle='-', linewidth=2, 
                label=f'Posterior Mean: ${tau_bayes:.2f}')
axes[1].axhline(TRUE_CAUSAL_EFFECT, color='green', linestyle='--', linewidth=2,
                label=f'True Effect: ${TRUE_CAUSAL_EFFECT:.2f}')
axes[1].set_xlabel('MCMC Iteration', fontsize=12)
axes[1].set_ylabel('Treatment Effect ($)', fontsize=12)
axes[1].set_title('MCMC Trace (Convergence Check)', fontsize=14, fontweight='bold')
axes[1].legend(fontsize=10)
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/bayesian_posterior.png', dpi=150, bbox_inches='tight')
plt.show()

---

## SECTION 7: Results Comparison

Now we compare all methods against the ground truth.

In [ ]:
# =============================================================================
# COMPREHENSIVE RESULTS COMPARISON
# =============================================================================

print("\n" + "="*80)
print("COMPREHENSIVE RESULTS COMPARISON")
print("="*80)

# Build results DataFrame
results_df = pd.DataFrame({
    'Method': ['Ground Truth'] + list(results.keys()),
    'Estimate': [TRUE_CAUSAL_EFFECT] + [r['estimate'] for r in results.values()],
    'Std Error': [0.0] + [r['se'] for r in results.values()],
    'CI Lower': [TRUE_CAUSAL_EFFECT] + [r['ci_lower'] for r in results.values()],
    'CI Upper': [TRUE_CAUSAL_EFFECT] + [r['ci_upper'] for r in results.values()],
    'Bias': [0.0] + [r['bias'] for r in results.values()],
    'Bias %': [0.0] + [(r['estimate']/TRUE_CAUSAL_EFFECT - 1)*100 for r in results.values()]
})

print(f"\nTRUE CAUSAL EFFECT: ${TRUE_CAUSAL_EFFECT:.2f}\n")
print(results_df.to_string(index=False))

# Save results
results_df.to_csv(RESULTS_OUTPUT_PATH, index=False)
print(f"\n✓ Results saved to: {RESULTS_OUTPUT_PATH}")

In [ ]:
# =============================================================================
# FOREST PLOT: Visual Comparison
# =============================================================================

fig, ax = plt.subplots(figsize=(12, 8))

methods = list(results.keys())
estimates = [results[m]['estimate'] for m in methods]
ci_lower = [results[m]['ci_lower'] for m in methods]
ci_upper = [results[m]['ci_upper'] for m in methods]

y_positions = np.arange(len(methods))
colors = ['red', 'steelblue', 'steelblue', 'purple']

# Plot estimates with confidence intervals
for i, (method, est, lower, upper, color) in enumerate(zip(methods, estimates, ci_lower, ci_upper, colors)):
    # Error bars
    ax.plot([lower, upper], [i, i], 'o-', linewidth=2.5, markersize=8, color=color)
    # Point estimate
    ax.plot(est, i, 'o', markersize=12, color=color, markeredgecolor='black', markeredgewidth=1.5)
    # Value label
    ax.text(upper + 0.5, i, f'${est:.2f}', va='center', fontsize=10, fontweight='bold')

# Ground truth line
ax.axvline(TRUE_CAUSAL_EFFECT, color='green', linestyle='--', linewidth=3, 
           label=f'Ground Truth: ${TRUE_CAUSAL_EFFECT:.2f}', zorder=0)

# Zero line
ax.axvline(0, color='black', linestyle=':', linewidth=1, alpha=0.5, zorder=0)

ax.set_yticks(y_positions)
ax.set_yticklabels(methods, fontsize=11)
ax.set_xlabel('Treatment Effect Estimate ($)', fontsize=13, fontweight='bold')
ax.set_title('Forest Plot: Method Comparison\n(Closer to green line = better)', 
             fontsize=15, fontweight='bold')
ax.grid(True, alpha=0.3, axis='x')
ax.legend(loc='upper right', fontsize=11)

plt.tight_layout()
plt.savefig(f'{PROJECT_DIR}/forest_plot_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

---

## SECTION 8: Conclusion

### Summary

In [ ]:
# =============================================================================
# FINAL SUMMARY
# =============================================================================

print("\n" + "="*80)
print("PIPELINE EXECUTION SUMMARY")
print("="*80)

print(f"\n📊 DATA GENERATION:")
print(f"   - Sample size: {N_SAMPLES}")
print(f"   - Data source: {'Gemini API' if USE_GEMINI_API else 'Templates'}")
print(f"   - Treatment rate: {df['program_participation'].mean()*100:.1f}%")

print(f"\n🔤 TEXT EMBEDDINGS:")
print(f"   - Model: {EMBEDDING_MODEL}")
print(f"   - Raw dimension: {embeddings_raw.shape[1]}")
print(f"   - PCA components used: {MAX_PCA_COMPONENTS}")

print(f"\n📈 CAUSAL ESTIMATION RESULTS:")
print(f"   True causal effect: ${TRUE_CAUSAL_EFFECT:.2f}")
print(f"   ")
for method, r in results.items():
    bias_pct = (r['estimate']/TRUE_CAUSAL_EFFECT - 1)*100
    status = "✗ BIASED" if abs(bias_pct) > 50 else "✓ Good" if abs(bias_pct) < 20 else "~ OK"
    print(f"   {method:20s}: ${r['estimate']:.2f} (bias: {bias_pct:+.1f}%) {status}")

print(f"\n🎯 KEY FINDINGS:")
print(f"   1. Naive OLS is SEVERELY BIASED due to omitted ability confounder")
print(f"   2. DML methods successfully reduce bias using text embeddings")
print(f"   3. Bayesian inference provides full uncertainty quantification")
print(f"   4. Text embeddings successfully proxy for latent ability")

print(f"\n💾 OUTPUT FILES:")
print(f"   - {DATA_OUTPUT_PATH}")
print(f"   - {EMBEDDINGS_OUTPUT_PATH}")
print(f"   - {RESULTS_OUTPUT_PATH}")
print(f"   - {PROJECT_DIR}/eda_confounding_analysis.png")
print(f"   - {PROJECT_DIR}/pca_validation.png")
print(f"   - {PROJECT_DIR}/bayesian_posterior.png")
print(f"   - {PROJECT_DIR}/forest_plot_comparison.png")

print("\n" + "="*80)
print("✓ PIPELINE COMPLETE!")
print("="*80)

In [ ]:
# =============================================================================
# SAVE FINAL ENHANCED DATASET
# =============================================================================

# Save the complete dataset with all results
final_output_path = f'{PROJECT_DIR}/data_with_embeddings_and_results.parquet'
df.to_parquet(final_output_path, index=False)

print(f"\n✓ Final dataset saved to: {final_output_path}")
print(f"  Shape: {df.shape}")
print(f"  Columns: {list(df.columns[:10])}...")

---

## Appendix: Quick Reference

### Key Configuration Parameters

| Parameter | Default | Description |
|-----------|---------|-------------|
| `N_SAMPLES` | 500 | Number of freelancers to generate |
| `USE_GEMINI_API` | True | Use Gemini API for realistic profiles |
| `EMBEDDING_MODEL` | 'all-MiniLM-L6-v2' | Pre-trained embedding model |
| `MAX_PCA_COMPONENTS` | 50 | Maximum PCA components to extract |
| `DML_N_PCA_COMPONENTS` | 20 | PCA components for DML |
| `BAYES_N_PCA_COMPONENTS` | 10 | PCA components for Bayesian |
| `MCMC_DRAWS` | 2000 | MCMC posterior draws |

### Troubleshooting

1. **Gemini API key not working**: Get a free key at https://aistudio.google.com/app/apikey
2. **Slow performance**: Reduce `N_SAMPLES` or set `USE_GEMINI_API = False`
3. **Memory errors**: Reduce `N_SAMPLES` or `MAX_PCA_COMPONENTS`
4. **Bayesian model slow**: Reduce `MCMC_DRAWS` or `BAYES_N_PCA_COMPONENTS`

### Further Reading

- [Double Machine Learning Paper](https://arxiv.org/abs/1608.00060)
- [Sentence Transformers Documentation](https://www.sbert.net/)
- [Bambi Documentation](https://bambinos.github.io/bambi/)
- [ArviZ Documentation](https://arviz-devs.github.io/arviz/)